# Formula 1 2025 Season Summary

## 1. Season Summary for Each Driver
## 2. The Championship Battle 
## 3. Key Races

### 1. Season Summary for Each Driver

Use plotly for an interactive graph

1. heatmap for each driver's result for each round
2. heatmap for total points

In [1]:
import fastf1 as ff1

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.io import show
from plotly.subplots import make_subplots

ff1.Cache.enable_cache('./data/cache')

In [2]:
season = 2025
schedule = ff1.get_event_schedule(season, include_testing=False)

schedule.head()

,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,...,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
1,1,Australia,Melbourne,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Australian Grand Prix,conventional,Practice 1,2025-03-14 12:30:00+11:00,2025-03-14 01:30:00,...,Practice 3,2025-03-15 12:30:00+11:00,2025-03-15 01:30:00,Qualifying,2025-03-15 16:00:00+11:00,2025-03-15 05:00:00,Race,2025-03-16 15:00:00+11:00,2025-03-16 04:00:00,True
2,2,China,Shanghai,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,2025-03-23,Chinese Grand Prix,sprint_qualifying,Practice 1,2025-03-21 11:30:00+08:00,2025-03-21 03:30:00,...,Sprint,2025-03-22 11:00:00+08:00,2025-03-22 03:00:00,Qualifying,2025-03-22 15:00:00+08:00,2025-03-22 07:00:00,Race,2025-03-23 15:00:00+08:00,2025-03-23 07:00:00,True
3,3,Japan,Suzuka,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,2025-04-06,Japanese Grand Prix,conventional,Practice 1,2025-04-04 11:30:00+09:00,2025-04-04 02:30:00,...,Practice 3,2025-04-05 11:30:00+09:00,2025-04-05 02:30:00,Qualifying,2025-04-05 15:00:00+09:00,2025-04-05 06:00:00,Race,2025-04-06 14:00:00+09:00,2025-04-06 05:00:00,True
4,4,Bahrain,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,2025-04-13,Bahrain Grand Prix,conventional,Practice 1,2025-04-11 14:30:00+03:00,2025-04-11 11:30:00,...,Practice 3,2025-04-12 15:30:00+03:00,2025-04-12 12:30:00,Qualifying,2025-04-12 19:00:00+03:00,2025-04-12 16:00:00,Race,2025-04-13 18:00:00+03:00,2025-04-13 15:00:00,True
5,5,Saudi Arabia,Jeddah,FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2025,2025-04-20,Saudi Arabian Grand Prix,conventional,Practice 1,2025-04-18 16:30:00+03:00,2025-04-18 13:30:00,...,Practice 3,2025-04-19 16:30:00+03:00,2025-04-19 13:30:00,Qualifying,2025-04-19 20:00:00+03:00,2025-04-19 17:00:00,Race,2025-04-20 20:00:00+03:00,2025-04-20 17:00:00,True


In [3]:
session_test = ff1.get_session(2025, 'Chinese Grand Prix', 'S')
session_test.load()

core           INFO 	Loading data for Chinese Grand Prix - Sprint [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

In [4]:
session_test.results.head()['Points']

44    8.0
81    7.0
1     6.0
63    5.0
16    4.0
Name: Points, dtype: float64

In [5]:
# Each driver's standing and points for each race
standings = []
event_names_short = [] # for the heatmap index later

for i, event in schedule.iterrows():
    event_name, round_number = event['EventName'], event['RoundNumber']
    event_names_short.append(event_name.replace("Grand Prix", ""))
    
    # load race data of that specific race
    session_r = ff1.get_session(season, event_name, "R")
    session_r.load()
    
    # add sprint race points if applicable
    # sprint race is called "sprint_qualifying" for EventFormat
    session_s = None
    if event['EventFormat'] == 'sprint_qualifying':
        session_s = ff1.get_session(season, event_name, 'S')
        session_s.load()
    
    # calculate points for each driver
    for j, driver_row in session_r.results.iterrows():
        abbr, race_points, race_position = driver_row['Abbreviation'], driver_row['Points'], driver_row['Position']
        
        sprint_points = 0
        if session_s is not None:
            driver_row = session_s.results.loc[session_s.results['Abbreviation'] == abbr] # keep rows that match the driver
            if not driver_row.empty:
                sprint_points = driver_row['Points'].values[0]
        
        standings.append({
            'EventName': event_name,
            'RoundNumber': round_number,
            'Driver': abbr,
            'Points': race_points + sprint_points,
            'Position': race_position
        })
    

core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

In [6]:
standings_df = pd.DataFrame(standings)
standings_df.head()

,EventName,RoundNumber,Driver,Points,Position
0,Australian Grand Prix,1,NOR,25.0,1.0
1,Australian Grand Prix,1,VER,18.0,2.0
2,Australian Grand Prix,1,RUS,15.0,3.0
3,Australian Grand Prix,1,ANT,12.0,4.0
4,Australian Grand Prix,1,ALB,10.0,5.0


__standings__ now contains: `EventName`, `RoundNumber`, `Driver`, and `Points`, `Position` for each race.

In [7]:
# drivers as rows, events as columns, use points for values
standings_df2 = standings_df.pivot(index='Driver', columns='RoundNumber', values='Points')
standings_df2.fillna(0)
standings_df2.head()

RoundNumber,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Driver,,,,,,,,,,,,,,,,,,,,,
ALB,10.0,6.0,2.0,0.0,2.0,10.0,10.0,2.0,0.0,0.0,...,10.0,6.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
ALO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,6.0,...,4.0,0.0,0.0,6.0,1.0,0.0,3.0,0.0,8.0,8.0
ANT,12.0,10.0,8.0,0.0,8.0,10.0,0.0,0.0,0.0,15.0,...,0.0,2.0,12.0,10.0,1.0,8.0,25.0,15.0,13.0,0.0
BEA,0.0,4.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,0.0,2.0,2.0,12.0,8.0,1.0,0.0,0.0
BOR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [8]:
standings_df2['total_points'] = standings_df2.sum(axis=1) # sum of points in row direction
standings_df2.sort_values(by='total_points', ascending=True, inplace=True) 
total_points = standings_df2['total_points'].values # store total_points data
standings_df2.drop(columns=['total_points']) # drop the col

positions = standings_df.pivot(index='Driver', columns='RoundNumber', values='Position').fillna('N/A')

In [9]:
total_points

array([  0.,   0.,  19.,  22.,  33.,  33.,  38.,  38.,  41.,  51.,  51.,
        56.,  64.,  73., 150., 156., 242., 319., 410., 421., 423.])

In [10]:
# hover text for plotly (got help from other materials)

# for each driver, for each round, extract that driver's position
hover_info = [
    [
        {
            'position': positions.at[driver, race]
        } for race in schedule['RoundNumber']
    ] for driver in standings_df2.index # because drivers are row indices
]

In [11]:
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.9, 0.1],
    subplot_titles=('2025 Summary', 'Total Points')
)
fig.update_layout(width=1600,
                  height=800,
                  paper_bgcolor='#222222',
                  font=dict(color='white')
                  )

# first heatmap
fig.add_trace( # add graph to a specific position
    go.Heatmap(
        x=event_names_short, 
        y=standings_df2.index, # drivers
        z=standings_df2.values, # points 
        texttemplate='%{text}', # displays point value on each cell
        text=standings_df2.values,
        customdata=hover_info,
        hovertemplate=(
            "Driver: %{y}<br>"
            "Race Name: %{x}<br>"
            "Points: %{z}<br>"
            "Position: %{customdata.position}<extra></extra>"
        ),
        colorscale='Pubu',
        showscale=False,
        zmin=0, # min color value starts at 0
        zmax=standings_df2.values.max() 
    ),
    row=1,
    col=1
)

# seocnd heatmap
fig.add_trace(
    go.Heatmap(
        x=['Total Points'] * len(total_points),
        y=standings_df2.index,
        z=total_points,
        texttemplate='%{text}',
        colorscale='Pubu',
        showscale=False,
        zmin=0,
        zmax=total_points.max()
    ),
    row=1,
    col=2
)

show(fig)